# Figure 1: OT ball versus bimodal HDR volume

This notebook compares a Euclidean ball in the source space with its kNN-smoothed optimal-transport pushforward under an exact POT EMD solve. The source is `N(0, I_2)` and the target is the equally weighted mixture

$$\frac{1}{2}N((-1, 0), I_2) + \frac{1}{2}N((1, 0), I_2).$$

The source radius is computed from the chi-squared quantile. The target HDR proxy is represented by two equal-radius balls centered at the target modes, with the radius calibrated by Monte Carlo under the bimodal target law. The exact discrete transport plan is converted to a barycentric map on source samples, then smoothed by averaging the mapped values of the `k` nearest source samples.

The exact OT cell uses `n_ot` source points and `n_ot` target points. It builds dense `n_ot x n_ot` cost and transport matrices, so it is the expensive cell in this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.spatial import cKDTree
from scipy.stats import chi2

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

seed = 7
rng = np.random.default_rng(seed)

# Probability mass of every plotted region.
coverage_mass = 0.25

n_ot = 20_000
n_radius_mc = 1_000_000
grid_size = 350
knn_smoothing_k = 50
n_boundary_points = 720

modes = np.array([[-1.0, 0.0], [1.0, 0.0]])
mode_separation = float(np.linalg.norm(modes[1] - modes[0]))

In [ ]:
def sample_source(n, rng):
    return rng.normal(size=(n, 2))


def sample_target_mixture(n, rng):
    component = rng.integers(0, len(modes), size=n)
    return rng.normal(size=(n, 2)) + modes[component], component


def nearest_mode_distances(y):
    return np.linalg.norm(y[:, None, :] - modes[None, :, :], axis=-1)


def hdr_component_masks(y, radius):
    distances = nearest_mode_distances(y)
    return distances[:, 0] <= radius, distances[:, 1] <= radius


def hdr_union_mask(y, radius):
    return np.min(nearest_mode_distances(y), axis=1) <= radius


def calibrate_hdr_radius(mass, n_mc, rng):
    y_mc, _ = sample_target_mixture(n_mc, rng)
    distances = np.min(nearest_mode_distances(y_mc), axis=1)
    radius = float(np.quantile(distances, mass, method="higher"))
    empirical_mass = float(np.mean(distances <= radius))
    return radius, empirical_mass


def circle_union_area(radius, distance):
    if radius <= 0.0:
        return 0.0
    if distance <= 0.0:
        return float(np.pi * radius**2)
    if 2.0 * radius <= distance:
        return float(2.0 * np.pi * radius**2)

    overlap = (
        2.0 * radius**2 * np.arccos(distance / (2.0 * radius))
        - 0.5 * distance * np.sqrt(max(4.0 * radius**2 - distance**2, 0.0))
    )
    return float(2.0 * np.pi * radius**2 - overlap)


def nearest_neighbor_grid(points, labels, bounds, size):
    xmin, xmax, ymin, ymax = bounds
    xs = np.linspace(xmin, xmax, size)
    ys = np.linspace(ymin, ymax, size)
    xx, yy = np.meshgrid(xs, ys)
    query = np.column_stack([xx.ravel(), yy.ravel()])

    tree = cKDTree(points)
    _, nearest = tree.query(query, workers=-1)
    zz = labels[nearest].reshape(xx.shape).astype(float)
    return xx, yy, zz


def grid_query_points(bounds, size):
    xmin, xmax, ymin, ymax = bounds
    xs = np.linspace(xmin, xmax, size)
    ys = np.linspace(ymin, ymax, size)
    xx, yy = np.meshgrid(xs, ys)
    query = np.column_stack([xx.ravel(), yy.ravel()])
    return xx, yy, query


def smoothed_ot_map(query_points, source_points, mapped_points, k):
    k = int(np.clip(k, 1, len(source_points)))
    tree = cKDTree(source_points)
    _, nearest = tree.query(query_points, k=k, workers=-1)
    if k == 1:
        return mapped_points[nearest]
    return mapped_points[nearest].mean(axis=1)


def smoothed_pullback_grids(
    source_points,
    mapped_points,
    bounds,
    size,
    hdr_radius,
    k,
):
    xx, yy, query = grid_query_points(bounds, size)
    mapped_query = smoothed_ot_map(query, source_points, mapped_points, k)
    left, right = hdr_component_masks(mapped_query, hdr_radius)
    union = left | right
    return (
        xx,
        yy,
        left.reshape(xx.shape).astype(float),
        right.reshape(xx.shape).astype(float),
        union.reshape(xx.shape).astype(float),
    )


def smoothed_pushforward_boundary(
    radius,
    source_points,
    mapped_points,
    k,
    n_points,
):
    theta = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=True)
    boundary = radius * np.column_stack([np.cos(theta), np.sin(theta)])
    return smoothed_ot_map(boundary, source_points, mapped_points, k)


def polygon_area(points):
    x = points[:, 0]
    y = points[:, 1]
    return float(0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def grid_area(mask_grid, bounds):
    xmin, xmax, ymin, ymax = bounds
    return float(np.mean(mask_grid > 0.5) * (xmax - xmin) * (ymax - ymin))


def mixture_density_grid(xx, yy):
    density = np.zeros_like(xx, dtype=float)
    for mode in modes:
        squared_radius = (xx - mode[0])**2 + (yy - mode[1])**2
        density += 0.5 * np.exp(-0.5 * squared_radius) / (2.0 * np.pi)
    return density


def source_density_grid(xx, yy):
    squared_radius = xx**2 + yy**2
    return np.exp(-0.5 * squared_radius) / (2.0 * np.pi)


def draw_binary_contour(ax, xx, yy, zz, **kwargs):
    if np.nanmin(zz) < 0.5 < np.nanmax(zz):
        return ax.contour(xx, yy, zz, levels=[0.5], **kwargs)
    return None

In [ ]:
r_source = float(np.sqrt(chi2.ppf(coverage_mass, df=2)))
r_hdr, hdr_radius_mc_mass = calibrate_hdr_radius(
    coverage_mass,
    n_radius_mc,
    rng,
)

X = sample_source(n_ot, rng)
Y, target_component = sample_target_mixture(n_ot, rng)

source_ball_mask = np.linalg.norm(X, axis=1) <= r_source
target_hdr_left, target_hdr_right = hdr_component_masks(Y, r_hdr)
target_hdr_union = target_hdr_left | target_hdr_right

print(f"coverage_mass = {coverage_mass:.3f}")
print(f"r_source from chi2 quantile = {r_source:.4f}")
print(f"r_hdr from MC calibration = {r_hdr:.4f}")
print(f"MC mass at r_hdr = {hdr_radius_mc_mass:.4f}")
print(f"source sample mass in ball = {source_ball_mask.mean():.4f}")
print(f"target sample mass in HDR balls = {target_hdr_union.mean():.4f}")

In [ ]:
a = np.full(n_ot, 1.0 / n_ot)
b = np.full(n_ot, 1.0 / n_ot)

M = np.ascontiguousarray(ot.dist(X, Y, metric="sqeuclidean"), dtype=np.float64)
G = ot.emd(a, b, M, numItermax=2_000_000, numThreads="max")

transport_cost = float(np.sum(G * M))
active_transport = G > 1e-12
max_row_support = int(active_transport.sum(axis=1).max())
max_col_support = int(active_transport.sum(axis=0).max())
del active_transport

source_mass = G.sum(axis=1)
discrete_ot_map = (G @ Y) / source_mass[:, None]

print(f"exact OT cost = {transport_cost:.4f}")
print(f"max source support size = {max_row_support}")
print(f"max target support size = {max_col_support}")
print(f"kNN smoothing k = {knn_smoothing_k}")

del M, G

In [ ]:
# source_limit = max(2., r_source + 1.10)
source_limit = 1.4
# target_limit_x = max(4.25, 1.0 + r_hdr + 1.10)
# target_limit_y = max(3.75, r_hdr + 1.10)
target_limit_x = 1.9
target_limit_y = 1.4

source_bounds = (-source_limit, source_limit, -source_limit, source_limit)
target_bounds = (-target_limit_x, target_limit_x, -target_limit_y, target_limit_y)

XXs, YYs, pull_left_grid, pull_right_grid, pull_hdr_grid = smoothed_pullback_grids(
    X,
    discrete_ot_map,
    source_bounds,
    grid_size,
    r_hdr,
    knn_smoothing_k,
)
pushforward_ball_contour = smoothed_pushforward_boundary(
    r_source,
    X,
    discrete_ot_map,
    knn_smoothing_k,
    n_boundary_points,
)

source_ball_volume = float(np.pi * r_source**2)
target_hdr_volume = circle_union_area(r_hdr, mode_separation)
pullback_hdr_volume = grid_area(pull_hdr_grid, source_bounds)
pushforward_ball_volume = polygon_area(pushforward_ball_contour)

volume_rows = [
    ("source ball B(0, r_coverage)", source_ball_volume),
    ("target HDR two-ball union", target_hdr_volume),
    ("smoothed pullback HDR union, grid estimate", pullback_hdr_volume),
    ("smoothed pushforward ball, polygon area", pushforward_ball_volume),
]

print("Volumes are 2D Lebesgue areas.")
for name, value in volume_rows:
    print(f"{name:<42} {value:8.3f}")

In [ ]:
fig, (ax_source, ax_target) = plt.subplots(
    1,
    2,
    figsize=(13.0, 6.0),
    constrained_layout=True,
)

# Pullback/source plot.
ax_source.add_patch(
    Circle(
        (0.0, 0.0),
        r_source,
        fill=False,
        edgecolor="forestgreen",
        linewidth=2.6,
    )
)
draw_binary_contour(
    ax_source,
    XXs,
    YYs,
    pull_left_grid,
    colors="firebrick",
    linewidths=2.2,
)
draw_binary_contour(
    ax_source,
    XXs,
    YYs,
    pull_right_grid,
    colors="firebrick",
    linewidths=2.2,
)
ax_source.set_title("Source Gaussian")
ax_source.set_xlabel(r"$y_1$")
ax_source.set_ylabel(r"$y_2$")
ax_source.set_xlim(source_bounds[0], source_bounds[1])
ax_source.set_ylim(source_bounds[2], source_bounds[3])
ax_source.set_aspect("equal", adjustable="box")
ax_source.legend(
    handles=[
        Line2D([0], [0], color="forestgreen", lw=2.6,
               label=r"$B(0, r_{\mathrm{coverage}})$"),
        Line2D([0], [0], color="firebrick", lw=2.2,
               label=r"$Q_k^{-1}($HDR$_{\mathrm{coverage}})$"),
        # Line2D([0], [0], color="firebrick"),
    ],
    loc="upper right",
    frameon=False,
)

# Pushforward/target plot.
ax_target.plot(
    pushforward_ball_contour[:, 0],
    pushforward_ball_contour[:, 1],
    color="forestgreen",
    linewidth=2.8,
)
for mode in modes:
    ax_target.add_patch(
        Circle(
            tuple(mode),
            r_hdr,
            fill=False,
            edgecolor="firebrick",
            linewidth=2.4,
        )
    )
    
ax_target.set_title("Target bimodal Gaussian")
ax_target.set_xlabel(r"$y_1$")
ax_target.set_ylabel(r"$y_2$")
ax_target.set_xlim(target_bounds[0], target_bounds[1])
ax_target.set_ylim(target_bounds[2], target_bounds[3])
ax_target.set_aspect("equal", adjustable="box")
ax_target.legend(
    handles=[
        Line2D([0], [0], color="forestgreen", lw=2.8,
               label=r"$Q_k(B(0, r_{\mathrm{coverage}}))$"),
        Line2D([0], [0], color="firebrick", lw=2.4,
               label=r"HDR$_{\mathrm{coverage}}$"),
    ],
    loc="upper right",
    frameon=False,
)

fig.suptitle(
    f"Q_k - kNN-smoothed Brenier map (k={knn_smoothing_k}), "
    + "coverage mass = "
    + f"{coverage_mass:.2f}",
    y=1.02,
)
plt.show()